# Coarse Fog Masking 實作與測試 (真實影片版)

這份 Notebook 實作了專案 Proposal 中定義的 **Coarse Fog Masking (粗略遮罩)**。利用局部對比度、HSV 飽和度以及影像梯度，透過聯集投票機制來找出初步的起霧像素。

**成員 A 的任務**：請將真實的起霧影片放入資料夾中，修改下方 Cell 的 `video_path`，並不斷微調 `thresh_var`, `thresh_sat`, `thresh_grad` 這三個參數，直到遮罩能完美捕捉到水氣且不誤判正常場景為止。

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 設定 matplotlib 顯示大小
plt.rcParams['figure.figsize'] = [18, 12]

part 1: coarse fog mask

### 核心函式定義

In [2]:
def get_coarse_fog_mask(image_bgr, patch_size=15, w_dcp=0.5, w_grad=0.3, w_sat=0.2):
    """
    基於物理先驗 (DCP, 梯度, 飽和度) 計算連續的霧氣信心分數。
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.float32) / 255.0
    
    # --- 特徵 1: 暗通道先驗 (DCP) ---
    min_channel = np.min(image_bgr.astype(np.float32) / 255.0, axis=2)
    kernel_dcp = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
    dark_channel = cv2.erode(min_channel, kernel_dcp)
    conf_dcp = dark_channel
    
    # --- 特徵 2: 梯度衰減 (Gradient) ---
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    grad_norm = np.clip(grad_mag / (np.percentile(grad_mag, 95) + 1e-6), 0, 1)
    conf_grad = 1.0 - grad_norm
    
    # --- 特徵 3: 飽和度衰減 (Saturation) ---
    sat = hsv[:, :, 1]
    conf_sat = 1.0 - sat
    
    # --- 信心分數加權融合 ---
    total_w = w_dcp + w_grad + w_sat
    w_dcp, w_grad, w_sat = w_dcp/total_w, w_grad/total_w, w_sat/total_w
    
    confidence_map = (w_dcp * conf_dcp) + (w_grad * conf_grad) + (w_sat * conf_sat)
    
    # --- 明度防呆 (選配) ---
    val = hsv[:, :, 2]
    confidence_map = confidence_map * np.clip(val * 2.0, 0, 1) 
    
    # 1. 大尺度高斯平滑 (模擬流體擴散)
    confidence_map = cv2.GaussianBlur(confidence_map, (101, 101), 0)
    
    # 2. 初始二值化 (Otsu's 或固定閾值)
    binary_mask = (confidence_map > 0.6).astype(np.uint8) * 255
    
    # 3. 形態學操作 (清理破碎點與孔洞)
    morph_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, morph_kernel)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, morph_kernel)
    
    return binary_mask, confidence_map

### 讀取真實影片與視覺化
請將 `video_path` 替換為你們手邊的影片路徑。你可以透過改變 `frame_id` 來隨機抽查影片中的不同時間點，確保參數在整支影片中都具備泛化能力。

part 2: homology transformation

In [3]:

def align_video_chunked(video_path, start_frame, end_frame, drift_threshold_ratio=0.4):
    """
    動態切割畫布對齊：當累積平移量過大時建立新區塊，並保留區塊間的邊界矩陣。
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ret, frame_prev = cap.read()
    if not ret: return []
        
    h, w = frame_prev.shape[:2]
    max_drift = min(w, h) * drift_threshold_ratio
    
    mask_prev, _ = get_coarse_fog_mask(frame_prev)
    valid_mask_prev = cv2.bitwise_not(mask_prev)
    gray_prev = cv2.cvtColor(frame_prev, cv2.COLOR_BGR2GRAY)
    
    sift = cv2.SIFT_create()
    kp_prev, des_prev = sift.detectAndCompute(gray_prev, mask=valid_mask_prev)
    bf = cv2.BFMatcher()
    
    chunks = []
    current_chunk = {
        'start_frame': start_frame,
        'end_frame': start_frame,
        'raw_H_list': [np.eye(3, dtype=np.float32)],
        'boundary_H': None  # 儲存上一區塊到本區塊的空間橋樑
    }
    
    H_accumulated = np.eye(3, dtype=np.float32)
    
    print(f"🔗 開始動態區塊對齊 (Drift Threshold: {max_drift:.1f} px)")
    for i in tqdm(range(start_frame + 1, end_frame + 1)):
        ret, frame_curr = cap.read()
        if not ret: break
            
        mask_curr, _ = get_coarse_fog_mask(frame_curr)
        valid_mask_curr = cv2.bitwise_not(mask_curr)
        gray_curr = cv2.cvtColor(frame_curr, cv2.COLOR_BGR2GRAY)
        kp_curr, des_curr = sift.detectAndCompute(gray_curr, mask=valid_mask_curr)
        
        matches = bf.knnMatch(des_curr, des_prev, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        
        H_curr_to_prev = None 
        if len(good_matches) > 15:
            src_pts = np.float32([kp_curr[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp_prev[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            affine_matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
            
            if affine_matrix is not None:
                # 🛡️ 強制剛體約束：防止連續旋轉導致畫布無限放大
                scale = np.sqrt(affine_matrix[0, 0]**2 + affine_matrix[1, 0]**2)
                affine_matrix[0:2, 0:2] /= scale 
                H_temp = np.eye(3, dtype=np.float32)
                H_temp[0:2, :] = affine_matrix
                H_curr_to_prev = H_temp
                
        if H_curr_to_prev is not None:
            H_accumulated = np.matmul(H_accumulated, H_curr_to_prev)
            kp_prev, des_prev = kp_curr, des_curr
            
        tx, ty = H_accumulated[0, 2], H_accumulated[1, 2]
        drift_distance = np.sqrt(tx**2 + ty**2)
        
        if drift_distance > max_drift:
            current_chunk['end_frame'] = i - 1
            chunks.append(current_chunk)
            
            boundary_H = H_curr_to_prev.copy() if H_curr_to_prev is not None else np.eye(3, dtype=np.float32)
            print(f"\n✂️ 幀數 {i}: 位移過大 ({drift_distance:.1f}px)！切割新畫布。")
            current_chunk = {
                'start_frame': i,
                'end_frame': i,
                'raw_H_list': [np.eye(3, dtype=np.float32)],
                'boundary_H': boundary_H
            }
            H_accumulated = np.eye(3, dtype=np.float32)
        else:
            current_chunk['raw_H_list'].append(H_accumulated.copy())
            current_chunk['end_frame'] = i
            
    chunks.append(current_chunk)
    cap.release()
    
    print("\n📐 計算區塊畫布尺寸...")
    for idx, chunk in enumerate(chunks):
        corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
        all_corners = np.concatenate([cv2.perspectiveTransform(corners, H) for H in chunk['raw_H_list']], axis=0)
        
        [x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
        [x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)
        
        chunk['pano_w'], chunk['pano_h'] = x_max - x_min, y_max - y_min
        T_offset = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)
        chunk['final_H_list'] = [np.matmul(T_offset, H) for H in chunk['raw_H_list']]
        print(f"  - Chunk {idx}: 畫布大小 {chunk['pano_w']}x{chunk['pano_h']}")

    return chunks

part 3: temporal fusion & repainting

In [4]:

def process_bidirectional_chunked_fusion(video_path, chunks):
    """
    四階段上帝視角去霧：獨立建圖 -> 正向傳播 -> 反向傳播 -> 最終渲染
    """
    cap = cv2.VideoCapture(video_path)
    
    # ==========================================
    # Phase 1: 獨立建圖 (每個 Chunk 各自建立初步背景)
    # ==========================================
    print("\n[Phase 1] 建立各區塊的獨立背景...")
    for chunk_idx, chunk in enumerate(chunks):
        pano_w, pano_h = chunk['pano_w'], chunk['pano_h']
        bg_sum = np.zeros((pano_h, pano_w, 3), dtype=np.float32)
        weight = np.zeros((pano_h, pano_w, 1), dtype=np.float32)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, chunk['start_frame'])
        for t in range(len(chunk['final_H_list'])):
            ret, frame = cap.read()
            if not ret: break
                
            fog_mask, _ = get_coarse_fog_mask(frame)
            clean_mask = (255 - fog_mask).astype(np.float32) / 255.0
            
            H = chunk['final_H_list'][t]
            warped_frame = cv2.warpPerspective(frame, H, (pano_w, pano_h)).astype(np.float32)
            warped_clean = np.expand_dims(cv2.warpPerspective(clean_mask, H, (pano_w, pano_h)), axis=-1)
            
            valid_mask = np.any(warped_frame > 0, axis=-1).astype(np.uint8)
            valid_mask = cv2.erode(valid_mask, np.ones((5, 5), np.uint8), iterations=3)
            valid_pixels = np.expand_dims(valid_mask, axis=-1).astype(np.float32)
            
            bg_sum += warped_frame * warped_clean * valid_pixels
            weight += warped_clean * valid_pixels
            
        chunk['bg_sum'] = bg_sum
        chunk['weight'] = weight

    # ==========================================
    # Phase 2: 正向時空穿越 (Forward Propagation)
    # 將過去的清晰背景投影到未來的畫布上
    # ==========================================
    print("\n[Phase 2] 執行正向時空記憶傳遞...")
    decay_factor = 0.8 # 歷史記憶的權重衰減
    
    for i in range(1, len(chunks)):
        prev_c, curr_c = chunks[i-1], chunks[i]
        if curr_c['boundary_H'] is None: continue
            
        # 計算 C1 -> C2 的空間轉換矩陣
        ret1, inv_prev_H_last = cv2.invert(prev_c['final_H_list'][-1])
        ret2, inv_bound_H = cv2.invert(curr_c['boundary_H'])
        if not (ret1 and ret2): continue
            
        curr_H_first = curr_c['final_H_list'][0]
        H_C1_to_C2 = curr_H_first @ inv_bound_H @ inv_prev_H_last
        
        # 變形並注入記憶
        warped_sum = cv2.warpPerspective(prev_c['bg_sum'], H_C1_to_C2, (curr_c['pano_w'], curr_c['pano_h']))
        warped_w = cv2.warpPerspective(prev_c['weight'], H_C1_to_C2, (curr_c['pano_w'], curr_c['pano_h']))
        warped_w = np.expand_dims(warped_w, axis=-1) if warped_w.ndim == 2 else warped_w
        
        curr_c['bg_sum'] += warped_sum * decay_factor
        curr_c['weight'] += warped_w * decay_factor

    # ==========================================
    # Phase 3: 反向時空穿越 (Backward Propagation)
    # 將未來發現的清晰背景倒轉回過去的畫布上
    # ==========================================
    print("\n[Phase 3] 執行反向時空記憶穿越 (解決開頭就有霧的問題)...")
    for i in range(len(chunks)-2, -1, -1):
        curr_c, next_c = chunks[i], chunks[i+1]
        if next_c['boundary_H'] is None: continue
            
        ret1, inv_curr_H_last = cv2.invert(curr_c['final_H_list'][-1])
        ret2, inv_bound_H = cv2.invert(next_c['boundary_H'])
        if not (ret1 and ret2): continue
            
        next_H_first = next_c['final_H_list'][0]
        H_C1_to_C2 = next_H_first @ inv_bound_H @ inv_curr_H_last
        ret3, H_C2_to_C1 = cv2.invert(H_C1_to_C2) # 反向穿越的關鍵
        if not ret3: continue
        
        warped_sum = cv2.warpPerspective(next_c['bg_sum'], H_C2_to_C1, (curr_c['pano_w'], curr_c['pano_h']))
        warped_w = cv2.warpPerspective(next_c['weight'], H_C2_to_C1, (curr_c['pano_w'], curr_c['pano_h']))
        warped_w = np.expand_dims(warped_w, axis=-1) if warped_w.ndim == 2 else warped_w
        
        curr_c['bg_sum'] += warped_sum * decay_factor
        curr_c['weight'] += warped_w * decay_factor

    # 結算所有畫布的最終背景圖
    for chunk in chunks:
        valid_bg = chunk['weight'] > 0
        clean_bg_ref = np.zeros_like(chunk['bg_sum'])
        clean_bg_ref[valid_bg.repeat(3, axis=-1)] = (chunk['bg_sum'] / (chunk['weight'] + 1e-6))[valid_bg.repeat(3, axis=-1)]
        chunk['final_bg'] = np.clip(clean_bg_ref, 0, 255).astype(np.uint8)

    # ==========================================
    # Phase 4: 最終渲染拉回 (Repainting)
    # ==========================================
    print("\n[Phase 4] 全知視角最終渲染...")
    final_dehazed_frames = []
    
    for chunk_idx, chunk in enumerate(chunks):
        cap.set(cv2.CAP_PROP_POS_FRAMES, chunk['start_frame'])
        clean_bg_ref = chunk['final_bg']
        prev_mask_float = None 
        
        for t in tqdm(range(len(chunk['final_H_list'])), desc=f"Chunk {chunk_idx}"):
            ret, frame_raw = cap.read()
            if not ret: break
                
            raw_h, raw_w = frame_raw.shape[:2]
            ret_inv, H_inv = cv2.invert(chunk['final_H_list'][t])
            if not ret_inv: H_inv = np.eye(3)
                
            bg_at_current_view = cv2.warpPerspective(clean_bg_ref, H_inv, (raw_w, raw_h))
            
            mask_coarse, _ = get_coarse_fog_mask(frame_raw) 
            mask_float = cv2.GaussianBlur(mask_coarse.astype(np.float32)/255.0, (81, 81), 0)
            
            if prev_mask_float is None: prev_mask_float = mask_float
            else:
                mask_float = 0.7 * prev_mask_float + 0.3 * mask_float 
                prev_mask_float = mask_float
                
            mask_3c = np.stack([mask_float]*3, axis=-1)
            
            # 取得有效背景遮罩並羽化
            valid_bg_2d = np.any(bg_at_current_view > 0, axis=-1).astype(np.uint8)
            valid_bg_2d = cv2.erode(valid_bg_2d, np.ones((5, 5), np.uint8), iterations=3)
            valid_bg_float = cv2.GaussianBlur(valid_bg_2d.astype(np.float32), (41, 41), 0)
            valid_bg_mask = np.expand_dims(valid_bg_float, axis=-1)
            
            final_mask = mask_3c * valid_bg_mask
            
            blended = frame_raw * (1.0 - final_mask) + bg_at_current_view * final_mask
            blended = np.clip(blended, 0, 255).astype(np.uint8)
            
            final_dehazed_frames.append(blended)
            
    cap.release()
    return final_dehazed_frames

In [5]:

# ==========================================
# 🚀 程式執行進入點
# ==========================================
if __name__ == "__main__":
    video_path = './data/foggy_vid/1.mp4'  # 請替換為您的影片路徑
    
    if not os.path.exists(video_path):
        print(f"找不到影片檔案：{video_path}")
    else:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        print(f"🎬 影片總幀數為: {total_frames}")

        # 1. 執行區塊對齊 (Drift 設為 0.4 可保持背景高解析度不發散)
        chunks = align_video_chunked(video_path, 0, total_frames - 1, drift_threshold_ratio=0.4)

        if chunks:
            # 2. 執行雙向時空穿越去霧融合
            dehazed_video = process_bidirectional_chunked_fusion(video_path, chunks)

            # 3. 儲存結果
            frame_h, frame_w = dehazed_video[0].shape[:2]
            out_path = 'dehazed_output_bidirectional.mp4'
            out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (frame_w, frame_h))
            for f in dehazed_video: out.write(f)
            out.release()
            
            print(f"\n🎉 大功告成！具有時空穿越記憶的高清去霧影片已儲存至：{out_path}")

🎬 影片總幀數為: 347
🔗 開始動態區塊對齊 (Drift Threshold: 192.0 px)


 24%|██▍       | 83/346 [00:08<00:27,  9.66it/s]


✂️ 幀數 82: 位移過大 (193.8px)！切割新畫布。


 39%|███▉      | 136/346 [00:14<00:22,  9.17it/s]


✂️ 幀數 135: 位移過大 (192.1px)！切割新畫布。


...


✂️ 幀數 325: 位移過大 (197.3px)！切割新畫布。


100%|██████████| 346/346 [00:36<00:00,  9.35it/s]



📐 計算區塊畫布尺寸...
  - Chunk 0: 畫布大小 678x872
  - Chunk 1: 畫布大小 684x865
  - Chunk 2: 畫布大小 676x866
  - Chunk 3: 畫布大小 678x893
  - Chunk 4: 畫布大小 668x867
  - Chunk 5: 畫布大小 682x871
  - Chunk 6: 畫布大小 550x859

[Phase 1] 建立各區塊的獨立背景...

[Phase 2] 執行正向時空記憶傳遞...

[Phase 3] 執行反向時空記憶穿越 (解決開頭就有霧的問題)...

[Phase 4] 全知視角最終渲染...


Chunk 6: 100%|██████████| 22/22 [00:01<00:00, 12.22it/s]



🎉 大功告成！具有時空穿越記憶的高清去霧影片已儲存至：dehazed_output_bidirectional.mp4
